# TRANSACTIONS: 
## EXECUTANDO BLOCOS DE OPERAÇÕES EM CONJUNTO VIA PYTHON

### Vamos criar um script em Python para realizar transações no MongoDB Atlas com o objetivo de:

- Registrar uma nova venda na coleção vendas.
- Subtrair a quantidade do produto vendido no estoque da coleção catalogoProdutos.
- Caso uma tentativa de venda de um produto exceda o estoque disponível, reverter a transação.

### Conectando o Python ao MongoDB Atlas

In [ ]:
# Importar bibliotecas
from pymongo import MongoClient # para criar sessão com o MongoDB
from datetime import datetime # Para inserir data atual como data da venda

# Pegar URI de conexão no MongoDB Atlas.
# Alterar <db_password> pela senha criada no capítulo anterior, tópico "COMO CRIAR UM CLUSTER GRATUITO PARA TESTE"
MONGODB_URI = "mongodb+srv://hashtagsql:<db_password>@cluster0.fxce4.mongodb.net/?retryWrites=true&w=majority&appName=Cluster0"

# Conectar
conexao = MongoClient(MONGODB_URI)
print("Conexão bem sucedida!")

### Pegando o nome do banco de dados e das coleções

In [ ]:
# Pega o nome do banco de dados
banco = conexao.hashCommerce

# Pega o nome das coleções
colecao_catalogo_produtos = banco.catalogoProdutos
colecao_vendas = banco.vendas

### Criando a função realizar_transacao()

In [ ]:
def realizar_transacao(session, produto_id, quantidade_vendida, dados_venda):
    
    # Verifica o estoque do produto
    produto = colecao_catalogo_produtos.find_one({"_id": produto_id}, session=session)
    
    if not produto:
        raise ValueError(f"Produto com ID {produto_id} não encontrado.")

    estoque_atual = produto["estoque"]

    if quantidade_vendida > estoque_atual:
        raise ValueError(f"Quantidade vendida ({quantidade_vendida}) excede o estoque disponível ({estoque_atual}).")

    # Subtrai a quantidade vendida do estoque
    colecao_catalogo_produtos.update_one(
        {"_id": produto_id},
        {"$inc": {"estoque": -quantidade_vendida}},
        session=session
    )
    
    # Insere a venda na coleção vendas
    colecao_vendas.insert_one(dados_venda, session=session)

### Criando a função processar_venda()

In [ ]:
def processar_venda(produto_id, quantidade_vendida, dados_venda):
    with conexao.start_session() as session:
        try:
            # Iniciar a transação da sessão
            session.start_transaction()

            # Realizar as operações da transação
            realizar_transacao(session, produto_id, quantidade_vendida, dados_venda)

            # Confirma a transação
            session.commit_transaction()
            print("Transação efetuada com sucesso!")
        except ValueError as e:
            # Se ocorrer um erro, reverte todas as operações da transação
            print(f"Erro: {e}")
            session.abort_transaction()
            print("Transação revertida.")

### Exemplo de venda de produto não cadastrado (transação não realizada)

In [ ]:
nova_venda = {
    "produto_id": 20,
    "data_venda": datetime.now(),
    "preco": 3499.00,
    "quantidade": 10,
    "desconto": 1000.00,
    "frete": None,
    "total": 33990.00,
    "forma_pagamento": "Pix",
    "status": "Concluído",
    "cliente": {
        "nome": "Carlos Ribeiro",
        "email": "carlosribeiro@gmail.com",
        "endereco": {
            "logradouro": "Rua Quatro",
            "numero": "123",
            "bairro": "Centro",
            "cidade": "Brasília",
            "estado": "DF",
            "CEP": "00000-000"
        }
    }
}

# Processar a venda (produto não cadastrado)
processar_venda(produto_id=nova_venda['produto_id'], quantidade_vendida=nova_venda['quantidade'], dados_venda=nova_venda)

### Exemplo de venda que quantidade excede o estoque (transação não realizada)

In [ ]:
nova_venda = {
    "produto_id": 15,
    "data_venda": datetime.now(),
    "preco": 3499.00,
    "quantidade": 100,
    "desconto": 1000.00,
    "frete": None,
    "total": 339900.00,
    "forma_pagamento": "Pix",
    "status": "Concluído",
    "cliente": {
        "nome": "Carlos Ribeiro",
        "email": "carlosribeiro@gmail.com",
        "endereco": {
            "logradouro": "Rua Quatro",
            "numero": "123",
            "bairro": "Centro",
            "cidade": "Brasília",
            "estado": "DF",
            "CEP": "00000-000"
        }
    }
}

# Processar a venda (estoque insuficiente)
processar_venda(produto_id=nova_venda['produto_id'], quantidade_vendida=nova_venda['quantidade'], dados_venda=nova_venda)

### Exemplo de nova venda bem-sucedida (transação realizada)

In [ ]:
nova_venda = {
    "produto_id": 15,
    "data_venda": datetime.now(),
    "preco": 3499.00,
    "quantidade": 10,
    "desconto": 1000.00,
    "frete": None,
    "total": 33990.00,
    "forma_pagamento": "Pix",
    "status": "Concluído",
    "cliente": {
        "nome": "Carlos Ribeiro",
        "email": "carlosribeiro@gmail.com",
        "endereco": {
            "logradouro": "Rua Quatro",
            "numero": "123",
            "bairro": "Centro",
            "cidade": "Brasília",
            "estado": "DF",
            "CEP": "00000-000"
        }
    }
}

# Processar a venda (produto cadastrado com estoque disponível)
processar_venda(produto_id=nova_venda['produto_id'], quantidade_vendida=nova_venda['quantidade'], dados_venda=nova_venda)

### Encerrando a conexão

In [ ]:
conexao.close()